# CP1 Week 11 -- Exceptions: Handling Errors

**Course:** Computer Programming 1 (CP1) | **Session:** 5 hours

## Learning Objectives

1. Handle errors gracefully with `try` / `except`
2. Recognize common exception types
3. Skip bad rows and count skip reasons
4. Make `clean_data()` report what was dropped and why

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: try / except Basics

Without error handling, ONE bad value crashes your entire pipeline.
With `try/except`, you catch the error, handle it, and keep going.

In [ ]:
def safe_float(text):
    """Convert text to float safely. Returns None if invalid."""
    try:
        return float(text)
    except (ValueError, TypeError):
        return None

test_values = ["42", "3.14", "abc", "", None, "  7  "]
for v in test_values:
    result = safe_float(v)
    print(f"  {str(v):>8} -> {result}")

**Expected Output:**
```
       42 -> 42.0
     3.14 -> 3.14
      abc -> None
          -> None
     None -> None
        7 -> 7.0
```

### Example 2 -- Without vs with error handling

In [ ]:
# WITHOUT error handling -- one bad value crashes everything:
data = ["10", "20", "abc", "30", "40"]

# This would crash on "abc":
# total = 0
# for item in data:
#     total += float(item)   # CRASH on "abc"!

# WITH error handling -- we skip bad values and keep going:
total = 0
count = 0
errors = 0
for item in data:
    try:
        total += float(item)
        count += 1
    except ValueError:
        errors += 1

if count > 0:
    print(f"Processed {count} values, skipped {errors}")
    print(f"Average: {total / count:.2f}")

**Expected Output:**
```
Processed 4 values, skipped 1
Average: 25.00
```

Without `try/except`, the third item ("abc") would have crashed the entire
program. With it, we skip the bad item and process the rest.

### Try It Yourself #1

In [ ]:
# TODO: Write a function safe_int(text) that:
# - Returns the integer value if text can be converted
# - Returns None if it cannot
# - Test with: "42", "3.14", "abc", "", None

def safe_int(text):
    pass  # your code here

for test in ["42", "3.14", "abc", "", None]:
    result = safe_int(test)
    print(f"  {str(test):>6} -> {result}")

---
## Part 2: Common Exception Types

In [ ]:
# ValueError
try:
    int("abc")
except ValueError as e:
    print(f"ValueError: {e}")

# KeyError
try:
    d = {"a": 1}
    print(d["b"])
except KeyError as e:
    print(f"KeyError: {e}")

# TypeError
try:
    "hello" + 5
except TypeError as e:
    print(f"TypeError: {e}")

# ZeroDivisionError
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"ZeroDivisionError: {e}")

---
## Part 3: Robust clean_data with Skip Reporting

In [ ]:
def clean_data_robust(data, config):
    """Clean data with detailed skip reporting."""
    cleaned = []
    report = {"missing": 0, "non_numeric": 0, "out_of_range": 0, "other": 0}

    min_val = config.get("min_value", float("-inf"))
    max_val = config.get("max_value", float("inf"))

    for i, row in enumerate(data):
        try:
            val = row.get("value")

            if val is None or str(val).strip() == "":
                report["missing"] += 1
                continue

            try:
                num = float(val)
            except (ValueError, TypeError):
                report["non_numeric"] += 1
                continue

            if num < min_val or num > max_val:
                report["out_of_range"] += 1
                continue

            cleaned.append({**row, "value": num})

        except Exception as e:
            report["other"] += 1
            print(f"  Unexpected error at row {i}: {e}")

    total_dropped = sum(report.values())
    print(f"Cleaning: {len(data)} raw -> {len(cleaned)} clean ({total_dropped} dropped)")
    for reason, count in report.items():
        if count > 0:
            print(f"  - {reason}: {count}")

    return cleaned, report

raw = [
    {"id": 1, "value": "25.0"},
    {"id": 2, "value": ""},
    {"id": 3, "value": "abc"},
    {"id": 4, "value": "500"},
    {"id": 5, "value": None},
    {"id": 6, "value": "42.0"},
]

config = {"min_value": 0, "max_value": 100}
cleaned, report = clean_data_robust(raw, config)
print(f"\nReport: {report}")

### Example -- Building a validation chain

In [ ]:
def validate_row(row, rules):
    """Run a list of validation rules on a row.
    Returns (is_valid, fail_reason or None).
    """
    for rule_name, rule_func in rules:
        try:
            if not rule_func(row):
                return False, rule_name
        except Exception as e:
            return False, rule_name + ": " + str(e)
    return True, None

# Define rules as functions
def has_value(row):
    return row.get("value") is not None and str(row.get("value")).strip() != ""

def is_numeric(row):
    float(row["value"])  # will raise ValueError if not numeric
    return True

def in_range(row):
    val = float(row["value"])
    return 0 <= val <= 100

rules = [
    ("missing_value", has_value),
    ("non_numeric", is_numeric),
    ("out_of_range", in_range),
]

# Test
test_rows = [
    {"id": 1, "value": "25"},
    {"id": 2, "value": ""},
    {"id": 3, "value": "abc"},
    {"id": 4, "value": "150"},
    {"id": 5, "value": "50"},
]

print("Validation Results:")
for row in test_rows:
    valid, reason = validate_row(row, rules)
    status = "PASS" if valid else f"FAIL ({reason})"
    print(f"  id={row['id']}, value={str(row['value']):>5} -> {status}")

**Expected Output:**
```
Validation Results:
  id=1, value=   25 -> PASS
  id=2, value=      -> FAIL (missing_value)
  id=3, value=  abc -> FAIL (non_numeric)
  id=4, value=  150 -> FAIL (out_of_range)
  id=5, value=   50 -> PASS
```

This pattern is powerful because you can add new rules without changing
the validation logic. Just add a new `(name, function)` pair to the list.

### Common Mistakes with Exceptions

| Mistake | What happens | Fix |
|---------|-------------|-----|
| Bare `except:` | Catches ALL errors including bugs | Use `except ValueError:` |
| Too broad try | Hides real bugs | Put only the risky line in try |
| Ignoring the error | Silent failures | At least log the error |
| Using exceptions for flow control | Slow and confusing | Use `if` checks first |

### Debugging Tip

When you catch an exception, always include `as e` so you can see the message:
```python
except ValueError as e:
    print(f"Error: {e}")  # Much better than just "pass"
```

---
## Key Takeaways -- Week 11

1. **`try/except`** catches errors so your program does not crash
2. **Specific exceptions** (ValueError, KeyError) are better than bare `except`
3. **Skip reporting** tells you WHY data was dropped
4. **Validation chains** make rules modular and extensible
5. **Robust pipelines** handle bad data gracefully without losing good data

---
## Homework

### Review (R1-R4)

In [ ]:
# R1: What does try/except do?
# R2: Name 4 common exception types.
# R3: Why is catching specific exceptions better than bare except?
# R4: What is skip reporting?

### Practice (P1-P5)

In [ ]:
# P1: Write safe_divide(a, b) that handles ZeroDivisionError.


In [ ]:
# P2: Write read_numbers(text_list) that converts strings to numbers,
# skipping invalid ones. Return (numbers, error_count).
data = ["10", "abc", "20.5", "", "30", "xyz", "40"]


In [ ]:
# P3: Update your clean_data() to track and report skip reasons.


In [ ]:
# P4: Write a safe file reader that returns a default if file missing.


In [ ]:
# P5: Write a function that tries multiple parsing strategies
# and returns the first one that succeeds.


### Challenge (C1-C2)

In [ ]:
# C1: Create a validation chain -- a list of check functions.
# If any fails, the row is dropped. Log which rule caused the drop.


In [ ]:
# C2: Write a retry decorator that retries a function N times on failure.


### Mini-Project

In [ ]:
# M1: Bulletproof Data Cleaner
# Write a cleaning function that handles EVERY possible error:
# - Missing values, non-numeric values, out-of-range values
# - Wrong types, None, empty strings, whitespace-only strings
# - NaN, infinity, negative zero
# Return (cleaned_data, detailed_report) where report has counts for each reason.
# Test with the most hostile data you can imagine.


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)